In [2]:
import os
import json
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset

# -------- Dataset：200次元特徴 + 相対速度 --------
class RelativeSpeedDataset200D(Dataset):
    def __init__(self, annot_root, distance_json_path, max_items=None):
        self.items = []
        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue
            sid = fname.replace(".json", "")
            if sid not in self.distances:
                continue

            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)
            seq = ann['sequence']
            if len(seq) < 20:
                continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            tgt = np.array([f['TgtSpeed_ref'] for f in seq], dtype=np.float32)
            keys = sorted(self.distances[sid].keys())
            if len(keys) < 20:
                continue
            dist = np.array([self.distances[sid][k] for k in keys], dtype=np.float32)

            def smooth(x, w):
                if len(x) < w:
                    return np.zeros_like(x)
                return np.convolve(x, np.ones(w)/w, mode='same')

            for i in range(len(seq) - 19):
                if max_items and len(self.items) >= max_items:
                    return

                d = dist[i:i+20]
                o = own[i:i+20]
                t = tgt[i:i+20]
                if np.any(np.isnan(d)) or np.any(np.isnan(o)) or np.any(np.isnan(t)):
                    continue

                rel_speed = t - o
                own_acc = np.gradient(o)
                d1 = np.gradient(d)
                d2 = np.gradient(d1)

                f3 = smooth(d, 3)
                f5 = smooth(d, 5)
                f7 = smooth(d, 7)
                f11 = smooth(d, 11)
                f11_d1 = np.gradient(f11) if len(f11) >= 3 else np.zeros_like(f11)

                try:
                    feat = np.concatenate([
                        d, o, own_acc, d1, d2,
                        f3[:20], f5[:20], f7[:20], f11[:20], f11_d1[:20]
                    ])
                except:
                    continue

                if feat.shape[0] != 200:
                    continue

                target = np.mean(rel_speed)
                self.items.append((feat.astype(np.float32), target, sid))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, tgt, sid = self.items[idx]
        return torch.tensor(feat), torch.tensor(tgt, dtype=torch.float32), sid

# -------- LSTMモデル：200D特徴を20x10に reshape --------
class LSTMRegressor200D(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=128, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = x.view(x.size(0), 20, 10)  # (B, 20, 10)
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]  # (B, hidden_dim)
        return self.fc(last_hidden).squeeze(1)

# -------- 学習ループ --------
def train_simple_model_200d(dataset, save_path="model513.pth"):
    scenes = sorted(set([item[-1] for item in dataset.items]))
    train_scenes, val_scenes = train_test_split(scenes, test_size=0.2, random_state=42)

    train_idx = [i for i, item in enumerate(dataset.items) if item[-1] in train_scenes]
    val_idx = [i for i, item in enumerate(dataset.items) if item[-1] in val_scenes]

    train_ds = Subset(dataset, train_idx)
    val_ds = Subset(dataset, val_idx)

    def collate_fn(batch):
        feats, tgts, sids = zip(*batch)
        return torch.stack(feats), torch.tensor(tgts), list(sids)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = LSTMRegressor200D().to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
    criterion = nn.SmoothL1Loss()

    best_val_loss = float('inf')
    patience = 20
    counter = 0

    for epoch in range(100):
        model.train()
        total_train_loss = 0
        for feats, tgts, _ in tqdm(train_loader, desc=f"[Train {epoch+1}]"):
            feats, tgts = feats.to(device), tgts.to(device)
            pred = model(feats)
            loss = criterion(pred, tgts)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * feats.size(0)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for feats, tgts, _ in val_loader:
                feats, tgts = feats.to(device), tgts.to(device)
                pred = model(feats)
                loss = criterion(pred, tgts)
                total_val_loss += loss.item() * feats.size(0)

        train_loss = total_train_loss / len(train_ds)
        val_loss = total_val_loss / len(val_ds)
        scheduler.step()

        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"✅ Saved model to {save_path} (val_loss={val_loss:.4f})")
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print(f"🛑 Early stopping at epoch {epoch+1}")
                break

    return model

# -------- 実行部 --------
if __name__ == "__main__":
    dataset = RelativeSpeedDataset200D(
        annot_root="../train/train_annotations",
        distance_json_path="../distance3/distance_estimates_corrected.json",
        max_items=7500
    )

    model = train_simple_model_200d(
        dataset,
        save_path="model513.pth"
    )

    print("✅ 学習完了: model_200d_lstm.pth に保存しました")


[Train 1]: 100%|██████████| 93/93 [00:00<00:00, 182.71it/s]


Epoch 1 | Train Loss: 3.0414 | Val Loss: 0.9574
✅ Saved model to model513.pth (val_loss=0.9574)


[Train 2]: 100%|██████████| 93/93 [00:00<00:00, 287.13it/s]


Epoch 2 | Train Loss: 0.6409 | Val Loss: 0.3518
✅ Saved model to model513.pth (val_loss=0.3518)


[Train 3]: 100%|██████████| 93/93 [00:00<00:00, 291.66it/s]


...

Epoch 43 | Train Loss: 0.0686 | Val Loss: 0.1912


[Train 44]: 100%|██████████| 93/93 [00:00<00:00, 293.80it/s]


Epoch 44 | Train Loss: 0.0589 | Val Loss: 0.2046


[Train 45]: 100%|██████████| 93/93 [00:00<00:00, 292.01it/s]

Epoch 45 | Train Loss: 0.0599 | Val Loss: 0.2128
🛑 Early stopping at epoch 45
✅ 学習完了: model_200d_lstm.pth に保存しました


In [3]:
import os
import json
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from tqdm import tqdm

# -------- LSTM モデル --------
class LSTMRegressor200D(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=128, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = x.view(x.size(0), 20, 10)  # (B, 20, 10)
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]  # (B, hidden)
        return self.fc(last_hidden).squeeze(1)

# -------- 推論用 Dataset --------
class InferenceDataset200D(Dataset):
    def __init__(self, annot_root, distance_json_path):
        self.items = []
        self.seq_lens = {}

        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue
            sid = fname.replace(".json", "")
            if sid not in self.distances:
                continue

            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)
            seq = ann['sequence']
            self.seq_lens[sid] = len(seq)

            if len(seq) < 20:
                continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            keys = sorted(self.distances[sid].keys())
            if len(keys) < 20:
                continue
            dist = np.array([self.distances[sid][k] for k in keys], dtype=np.float32)

            def smooth(x, w):
                return np.convolve(x, np.ones(w)/w, mode='same') if len(x) >= w else np.zeros_like(x)

            for i in range(len(seq) - 19):
                d = dist[i:i+20]
                o = own[i:i+20]
                if np.any(np.isnan(d)) or np.any(np.isnan(o)):
                    continue

                own_acc = np.gradient(o)
                d1 = np.gradient(d)
                d2 = np.gradient(d1)

                f3 = smooth(d, 3)
                f5 = smooth(d, 5)
                f7 = smooth(d, 7)
                f11 = smooth(d, 11)
                f11_d1 = np.gradient(f11) if len(f11) >= 3 else np.zeros_like(f11)

                feat = np.concatenate([
                    d, o, own_acc, d1, d2,
                    f3[:20], f5[:20], f7[:20], f11[:20], f11_d1[:20]
                ])

                if feat.shape[0] != 200:
                    continue

                own_avg = np.mean(o)
                self.items.append((feat.astype(np.float32), own_avg, sid, i))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, own_avg, sid, frame_idx = self.items[idx]
        return torch.tensor(feat), own_avg, sid, frame_idx

# -------- 推論処理 --------
def predict_and_save_submission(
    model_path,
    annot_root,
    distance_json_path,
    save_path="submission.json"
):
    dataset = InferenceDataset200D(annot_root, distance_json_path)
    loader = DataLoader(dataset, batch_size=64, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = LSTMRegressor200D().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    raw_preds = defaultdict(list)
    with torch.no_grad():
        for feats, own_speeds, sids, frame_idxs in tqdm(loader):
            feats = feats.to(device)
            preds = model(feats).cpu().numpy()
            own_speeds = own_speeds.numpy()
            abs_speeds = preds + own_speeds  # 相対速度 + 自車速度

            for sid, frame_idx, tgt in zip(sids, frame_idxs, abs_speeds):
                raw_preds[sid].append((frame_idx + 19, float(round(tgt, 3))))  # 20フレーム目の出力

    submission = {}
    for sid, pairs in raw_preds.items():
        pairs.sort()
        seq_len = dataset.seq_lens.get(sid, max(f for f, _ in pairs) + 1)
        pred_list = [0.0] * seq_len
        for idx, val in pairs:
            if idx < seq_len:
                pred_list[idx] = val
        for i in range(1, seq_len):
            if pred_list[i] == 0.0:
                pred_list[i] = pred_list[i-1]
        submission[sid] = pred_list

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)

    print(f"✅ 完成: {save_path} に保存しました（scene数: {len(submission)}）")

# -------- 実行部 --------
if __name__ == "__main__":
    predict_and_save_submission(
        model_path="model513.pth",
        annot_root="../test/test_annotations",
        distance_json_path="../testdistance/testdistance_estimates_smoothed.json",
        save_path="submission.json"
    )


100%|██████████| 395/395 [00:01<00:00, 312.56it/s]


✅ 完成: submission.json に保存しました（scene数: 239）
